In [2]:
#  Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

#  Load Dataset
df = pd.read_csv(r"C:\\Users\\ma007\\Downloads\\archive (8)\\student_exam_scores.csv")
print("Dataset Loaded Successfully ")
display(df.head())

# Basic Info
print(df.info())
print(df.describe())

# Feature & Target Split
# Drop 'student_id' since it's not a predictive feature
X = df.drop(['student_id', 'exam_score'], axis=1)
y = df['exam_score']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialization of Random Forest Model
rf = RandomForestRegressor(random_state=42)

# K-Fold Cross Validation (Before Tuning)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X_train, y_train, cv=kf, scoring='r2')
print(f"\nCross-Validation R² Scores (Before Tuning): {cv_scores}")
print(f"Mean R² (Before Tuning): {cv_scores.mean():.4f}")

# Train & Evaluate Before Tuning
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print("\nBefore Tuning Results:")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

# Hyperparameter Tuning using GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(estimator=rf, param_grid=param_grid,
                           cv=5, n_jobs=-1, scoring='r2', verbose=2)

grid_search.fit(X_train, y_train)
print("\nBest Parameters from Grid Search:")
print(grid_search.best_params_)

# Evaluation After Tuning
best_rf = grid_search.best_estimator_
best_rf.fit(X_train, y_train)
y_pred_tuned = best_rf.predict(X_test)

print("\nAfter Tuning Results:")
print(f"R² Score: {r2_score(y_test, y_pred_tuned):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_tuned)):.4f}")

# K-Fold Cross Validation (After Tuning)
cv_scores_tuned = cross_val_score(best_rf, X_train, y_train, cv=kf, scoring='r2')
print(f"\nCross-Validation R² Scores (After Tuning): {cv_scores_tuned}")
print(f"Mean R² (After Tuning): {cv_scores_tuned.mean():.4f}")


Dataset Loaded Successfully 


,student_id,hours_studied,sleep_hours,attendance_percent,previous_scores,exam_score
0,S001,8.0,8.8,72.1,45,30.2
1,S002,1.3,8.6,60.7,55,25.0
2,S003,4.0,8.2,73.7,86,35.8
3,S004,3.5,4.8,95.1,66,34.0
4,S005,9.1,6.4,89.8,71,40.3


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   student_id          200 non-null    object 
 1   hours_studied       200 non-null    float64
 2   sleep_hours         200 non-null    float64
 3   attendance_percent  200 non-null    float64
 4   previous_scores     200 non-null    int64  
 5   exam_score          200 non-null    float64
dtypes: float64(4), int64(1), object(1)
memory usage: 9.5+ KB
None
       hours_studied  sleep_hours  attendance_percent  previous_scores  \
count     200.000000   200.000000          200.000000       200.000000   
mean        6.325500     6.622000           74.830000        66.800000   
std         3.227317     1.497138           14.249905        15.663869   
min         1.000000     4.000000           50.300000        40.000000   
25%         3.500000     5.300000           62.200000        54.000

# Reflective Summary

In [5]:
'''##  Reflection: Impact of Hyperparameter Tuning on Random Forest Model

For this experiment, I used the **Student Exam Scores** dataset to predict students’ exam performance using study-related factors such as study hours, sleep duration, attendance, and previous scores.

###  Model Setup
- **Algorithm:** Random Forest Regressor  
- **Validation:** 5-Fold Cross Validation  
- **Tuning Method:** GridSearchCV  

###  Performance Summary

| Metric | Before Tuning | After Tuning | Improvement |
|:-------|:--------------:|:-------------:|:------------:|
| Mean Cross-Validation R² | 0.7397 | 0.7414 | +0.0017 |
| Test R² | 0.7922 | 0.8051 | +0.0129 |
| RMSE | 3.3206 | 3.2164 | ↓ 0.1042 |

###  Insights
- The tuned model slightly outperformed the baseline, improving both **R²** and **RMSE**, which indicates more accurate predictions.  
- The improvement may seem minor, but it demonstrates that even modest hyperparameter adjustments—especially to `max_depth` and `n_estimators`—can enhance generalization and reduce overfitting.  
- The optimal parameters found (`max_depth=10`, `min_samples_leaf=4`, `n_estimators=300`) suggest that a balanced tree depth and moderate leaf size helped stabilize model variance.

###  Conclusion
Overall, **hyperparameter tuning improved model consistency and predictive reliability**.  
The model now performs more uniformly across folds, showing it can generalize better to unseen student data.
'''


'##  Reflection: Impact of Hyperparameter Tuning on Random Forest Model\n\nFor this experiment, I used the **Student Exam Scores** dataset to predict students’ exam performance using study-related factors such as study hours, sleep duration, attendance, and previous scores.\n\n###  Model Setup\n- **Algorithm:** Random Forest Regressor  \n- **Validation:** 5-Fold Cross Validation  \n- **Tuning Method:** GridSearchCV  \n\n###  Performance Summary\n\n| Metric | Before Tuning | After Tuning | Improvement |\n|:-------|:--------------:|:-------------:|:------------:|\n| Mean Cross-Validation R² | 0.7397 | 0.7414 | +0.0017 |\n| Test R² | 0.7922 | 0.8051 | +0.0129 |\n| RMSE | 3.3206 | 3.2164 | ↓ 0.1042 |\n\n###  Insights\n- The tuned model slightly outperformed the baseline, improving both **R²** and **RMSE**, which indicates more accurate predictions.  \n- The improvement may seem minor, but it demonstrates that even modest hyperparameter adjustments—especially to `max_depth` and `n_estimator